In [1]:
import pandas as pd
import numpy as np
import gc

from pipeline.preprocessing import reduce_memory_usage

### Data Load

In [ ]:
# 데이터 경로
base_dir = './open'

# 데이터 분할(폴더) 구분
data_splits = ["train"] # train 만!

# 각 데이터 유형별 폴더명, 파일 접미사, 변수 접두어 설정
data_categories = {
    "청구정보": {"folder": "4.청구입금정보", "suffix": "청구정보", "var_prefix": "billing"}
}

# 2018년 7월부터 12월까지의 월 리스트
months = ['07', '08', '09', '10', '11', '12']

for split in data_splits:
    for category, info in data_categories.items():
        folder = info["folder"]
        suffix = info["suffix"]
        var_prefix = info["var_prefix"]

        for month in months:
            # 파일명 형식: 2018{month}_{split}_{suffix}.parquet
            file_path = f"{base_dir}/{split}/{folder}/2018{month}_{split}_{suffix}.parquet"
            # 변수명 형식: {var_prefix}_{split}_{month}
            variable_name = f"{var_prefix}_{split}_{month}"
            globals()[variable_name] = pd.read_parquet(file_path)
            print(f"{variable_name} is loaded from {file_path}")

gc.collect()

billing_train_07 is loaded from ./open/train/4.청구입금정보/201807_train_청구정보.parquet
billing_train_08 is loaded from ./open/train/4.청구입금정보/201808_train_청구정보.parquet
billing_train_09 is loaded from ./open/train/4.청구입금정보/201809_train_청구정보.parquet
billing_train_10 is loaded from ./open/train/4.청구입금정보/201810_train_청구정보.parquet
billing_train_11 is loaded from ./open/train/4.청구입금정보/201811_train_청구정보.parquet
billing_train_12 is loaded from ./open/train/4.청구입금정보/201812_train_청구정보.parquet


1130

In [4]:
# 데이터 유형별 설정
info_categories = ["billing"]

# 월 설정
months = ['07', '08', '09', '10', '11', '12']

# 각 유형별로 월별 데이터를 합쳐서 새로운 변수에 저장
train_dfs = {}

for prefix in info_categories:
    # globals()에서 동적 변수명으로 데이터프레임들을 가져와 리스트에 저장
    df_list = [globals()[f"{prefix}_train_{month}"] for month in months]
    train_dfs[f"{prefix}_train_df"] = pd.concat(df_list, axis=0)
    gc.collect()
    print(f"{prefix}_train_df is created with shape: {train_dfs[f'{prefix}_train_df'].shape}")

billing_train_df  = train_dfs["billing_train_df"]

gc.collect()

billing_train_df is created with shape: (2400000, 46)


0

In [ ]:
# 대표결제일이 10인 경우: D, E만 있다 -> 대표결제일 10 여부 변수 추가
billing_train_df['대표결제일_10여부'] = np.where(billing_train_df['대표결제일'] == 10, 1, 0)

# 대표결제일이 21인 경우: C, D, E만 있다 -> 대표결제일 21 여부 변수 추가
billing_train_df['대표결제일_21여부'] = np.where(billing_train_df['대표결제일'] == 21, 1, 0)

billing_train_df = billing_train_df.drop(['대표결제일'], axis=1) #원래 대표결제일 변수 삭제

# 청구서수령방법이 당사멤버십인 고객: C, D, E만 있다 -> 청구서수령방법 당사멤버십 여부 변수 추가
billing_train_df['청구서수령방법_당사멤버십여부'] = np.where(billing_train_df['청구서수령방법'] == '당사멤버십', 1, 0)
billing_train_df = billing_train_df.drop(['청구서수령방법'], axis=1) # 원래 청구서수령방법 변수 삭제

# 대표결제방법코드: 모두 '자동이체'로 동일 -> 삭제
# 대표청구서수령지구분코드, 대표청구지고객주소구분코드: 청구서수령방법에 대해 세분화한 것 -> 삭제
# 청구서발송여부_B0, 청구서발송여부_R3M, 청구서발송여부_R6M: 0,1로 구성되어있으며 청구금액이 0이면 미발송 -> 청구금액 변수와 겹치므로 삭제
billing_train_df = billing_train_df.drop(['대표결제방법코드', '대표청구서수령지구분코드', '대표청구지고객주소구분코드', '청구서발송여부_B0',
                                          '청구서발송여부_R3M', '청구서발송여부_R6M'], axis=1)

# 문자열 인코딩
billing_train_df['할인건수_R3M'] = billing_train_df['할인건수_R3M'].map({'1회 이상': 0, '10회 이상': 1, '20회 이상': 2,'30회 이상': 3, '40회 이상': 4})
billing_train_df['할인건수_B0M'] = billing_train_df['할인건수_B0M'].map({'1회 이상': 0, '10회 이상': 1})

In [6]:
billing_train_df['청구서발송여부_all'] = ((billing_train_df['청구서발송여부_B0'] == 1) & 
                                        (billing_train_df['청구서발송여부_R3M'] == 1) & 
                                        (billing_train_df['청구서발송여부_R6M'] == 1)
                                        ).astype(int)

In [ ]:
# billing_train_df = reduce_memory_usage(billing_train_df)

# billing_train_df.to_parquet("../data/Dacon/billing_train_cleaned.parquet")

In [ ]:
# 1차 전처리 끝난 train 데이터 컬럼명 csv 파일 생성
train_df = pd.read_parquet('./data/Data/train_df_cleaned.parquet')
pd.DataFrame({'feature': train_df.columns}).to_csv('./data/Data/train_df_cleaned_feature_list.csv', index=False, encoding='utf-8-sig')